# Drug Review Insights & Summarization Tool: Data Ingestion and Preprocessing

**Dataset:** UCI ML Drug Review Dataset (Drugs.com) — 53,800 patient reviews covering 2,637 unique drugs across 708 conditions, with 10-star ratings, free-text reviews, and helpfulness votes spanning 2008–2017.

**Kaggle:** https://www.kaggle.com/datasets/jessicali9530/kuc-hackathon-winter-2018 

**UCI Repository:** https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com 

## Data & Preprocessing

This notebook prepares the Drug Review dataset for downstream sentiment analysis, visualization, and GenAI summarization. The cleaned output is designed to be used by the main project pipeline and later modules.

Notebook: `preprocessing/drug_review_preprocessing.ipynb`

This notebook:
- Loads the Drug Review dataset
- Checks shape, columns, data types, missing values, and duplicates
- Cleans key fields such as `drugName`, `condition`, `review`, `rating`, `date`, and `usefulCount`
- Converts dates and numeric fields to the correct data types
- Creates preprocessing features such as `review_length`, `review_word_count`, `review_year`, and `rating_sentiment`
- Generates summary statistics and distribution plots
- Exports a cleaned dataset for downstream GenAI summarization and visualization

Output files:
- `preprocessing/cleaned_drug_reviews.csv`
- `preprocessing/drug_review_profile.csv`
- `preprocessing/figures/`

# Import Packages 

In [1]:
import kagglehub
import os   
import pandas as pd
from sklearn.model_selection import train_test_split

/Users/nancywalker/Documents/University_SanDiego/ADS-507PracticalDataEngineering/ADS-507-Final-Team-Project/.conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Install ucimlrepo package if not already installed 
# !pip install ucimlrepo
from ucimlrepo import fetch_ucirepo 

## Load and Ingest Datasets 

- Read the UCI/Kaggle CSV files into pandas.
- Confirm row counts, columns, data types, and missing values.

In [3]:
# Download latest version
path = kagglehub.dataset_download("jessicali9530/kuc-hackathon-winter-2018")

print("Path to dataset files:", path)

Path to dataset files: /Users/nancywalker/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2


In [4]:
# Check that datasets are present
print("Files in dataset directory:", os.listdir(path))

# Load the datasets
train_df = pd.read_csv(
    os.path.join(path, "drugsComTrain_raw.csv")
)
test_df = pd.read_csv(
    os.path.join(path, "drugsComTest_raw.csv")
)

print("Train dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

Files in dataset directory: ['drugsComTrain_raw.csv', 'drugsComTest_raw.csv']
Train dataset shape: (161297, 7)
Test dataset shape: (53766, 7)


This project is already split into a traing and testing dataset. To create our own split parameters after merging with other datasets will combine data then split. All preprocessingsteps that learns from the data (TF-IDF, scaling, embeddings normalization, SMOTE, etc.) are to be performed after splitting to avoid data leakage. 

In [6]:
# Save datasets to raw data directory for preprocessing
train_df.to_csv("../data/raw/drugsComTrain_raw.csv", index=False)
test_df.to_csv("../data/raw/drugsComTest_raw.csv", index=False)

In [9]:
# Combine original train and test
kaggle_full_df = pd.concat([train_df, test_df], ignore_index=True)
print("Full dataset shape:", full_df.shape)

Full dataset shape: (215063, 7)


In [10]:
# Save combined dataset for preprocessing
kaggle_full_df.to_csv(
    "../data/processed/kaggle_combined.csv",
    index=False
)

In [11]:
# Check datatypes and missing values
print("Data types:\n", kaggle_full_df.dtypes)
print("\nMissing values:\n", kaggle_full_df.isnull().sum())

Data types:
 uniqueID        int64
drugName       object
condition      object
review         object
rating          int64
date           object
usefulCount     int64
dtype: object

Missing values:
 uniqueID          0
drugName          0
condition      1194
review            0
rating            0
date              0
usefulCount       0
dtype: int64


In [12]:
# View the percentage of missing values in each column
missing_percent = (kaggle_full_df.isnull().sum() / len(kaggle_full_df)) * 100
print("\nPercentage of missing values:\n", missing_percent)


Percentage of missing values:
 uniqueID       0.000000
drugName       0.000000
condition      0.555186
review         0.000000
rating         0.000000
date           0.000000
usefulCount    0.000000
dtype: float64


Important columns for text analysis have zero missing values. For example, review and rating. 

Condition has 0.56% columns with missing values. Since this column is not the target of analysis, ,issing rows can be filled with Unknown. 

In [13]:
# Fill missing condition values with "Unknown"
kaggle_full_df["condition"] = kaggle_full_df["condition"].fillna("Unknown")

# Check misisng value counts 
print("\nMissing values:\n", kaggle_full_df.isnull().sum())


Missing values:
 uniqueID       0
drugName       0
condition      0
review         0
rating         0
date           0
usefulCount    0
dtype: int64


In [14]:
kaggle_full_df.head()

,uniqueID,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,20-May-12,27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,27-Apr-10,192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,14-Dec-09,17
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,3-Nov-15,10
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,27-Nov-16,37


In [15]:
# Save cleaned dataset for preprocessing
kaggle_full_df.to_csv("../data/processed/kaggle_full_cleaned.csv", index=False)

- Read the UCI/Druglib CSV files into pandas.
- Confirm row counts, columns, data types, and missing values.

In [16]:
# fetch dataset 
drug_reviews_druglib_com = fetch_ucirepo(id=461) 
  
# data (as pandas dataframes) 
X = drug_reviews_druglib_com.data.features 
y = drug_reviews_druglib_com.data.targets 
  
# metadata 
print(drug_reviews_druglib_com.metadata) 
  
# variable information 
print(drug_reviews_druglib_com.variables) 

{'uci_id': 461, 'name': 'Drug Reviews (Druglib.com)', 'repository_url': 'https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com', 'data_url': 'https://archive.ics.uci.edu/static/public/461/data.csv', 'abstract': 'The dataset provides patient reviews on specific drugs along with related conditions. Reviews and ratings are grouped into reports on the three aspects benefits, side effects and overall comment.', 'area': 'Health and Medicine', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate', 'Text'], 'num_instances': 4143, 'num_features': 8, 'feature_types': ['Integer'], 'demographics': [], 'target_col': None, 'index_col': ['reviewID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2018, 'last_updated': 'Wed Apr 03 2024', 'dataset_doi': '10.24432/C55G6J', 'creators': ['Surya Kallumadi', 'Felix Grer'], 'intro_paper': {'ID': 457, 'type': 'NATIVE', 'title': 'Aspect-Based Sentiment Analysis of D

The UCI Drug Review dataset contains no missing values. 

In [20]:
# Convert UCI features into dataframe
druglib_df = X.copy()

Sometimes in the UCI dataset:

rating may already exist in X
and y could instead contain effectiveness

So before adding rating = y, run:

In [22]:
print(X.columns)
print(type(y))
print(y)

Index(['urlDrugName', 'rating', 'effectiveness', 'sideEffects', 'condition',
       'benefitsReview', 'sideEffectsReview', 'commentsReview'],
      dtype='object')
<class 'NoneType'>
None


All variable are already inside of X 

In [23]:
# Save Raw data after download/load
druglib_df.to_csv("../data/raw/druglib_df.csv", index=False)

In [24]:
# Preview
druglib_df.head()

,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


In [26]:
# Save individual datasets for preprocessing
druglib_df.to_csv("../data/processed/druglib_full.csv", index=False)

### Create a Unified Schema

In [27]:
# Rename Columns for consistency
druglib_df = druglib_df.rename(columns={
    "urlDrugName": "drugName"
})

# Combine the three review text columns 
druglib_df["review"] = (
    druglib_df["benefitsReview"].fillna('') + " " +
    druglib_df["sideEffectsReview"].fillna('') + " " +
    druglib_df["commentsReview"].fillna('')
)

In [28]:
# keep shared columns for merging
druglib_subset = druglib_df[
    ["drugName", "condition", "review", "rating"]
]

# Match kaggle column names for merging
kaggle_subset = full_df[
    ["drugName", "condition", "review", "rating"]
]

In [29]:
# Combine the two datasets
combined_df = pd.concat(
    [kaggle_subset, druglib_subset],
    ignore_index=True
)

In [30]:
# Save combined dataset for preprocessing
combined_df.to_csv(
    "../data/processed/kaggle_druglib_combined.csv",
    index=False
)

## Clean The Data 

- Handle missing values in condition, review, rating, date, usefulCount.
- Remove duplicates.
- Clean review text lightly, but do not over-clean because the GenAI model needs readable patient language.
- Standardize column names.
- Convert date to datetime.
- Make sure ratings are numeric.

## Create Useful Features 

- Review length
- Sentiment label from rating, such as:
    - 1–4 = negative
    - 5–6 = neutral
    - 7–10 = positive
- Year from date
- Possibly helpfulness-weighted rating

Sentiment analysis: What elements of a review make it more helpful to others? Which patients tend to have more negative reviews? Can you determine if a review is positive, neutral, or negative?


## Output a structured data profile

- Summary statistics
- Missing value table
- Rating distribution
- Top drugs
- Top conditions
- Review length distribution
- Positive/negative review counts by condition or drug

## Save cleaned outputs

- cleaned_drug_reviews.csv
- drug_review_profile.csv
- visualizations as .png if needed

target_column: 
- sentiment label
- misinformation label
- condition category
- adverse event flag
- rating class